# 💼 Trabalho Final — Alocação de Desenvolvedores em Projetos
**Pesquisa Operacional e Otimização em IA** — MBA em Ciência de Dados (UNIFOR)

Prof. Mafra | mafra@verboo.ai

---

O departamento de **IA** está organizando a alocação de sua equipe de desenvolvimento para o próximo ciclo. São **7 desenvolvedores** e **5 projetos** ativos, e o objetivo é encontrar a alocação que **maximize a produtividade total** da equipe, respeitando um conjunto de restrições operacionais.

In [8]:
!pip install pulp -q
from pulp import *


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


---
## Contexto

A equipe é composta por **7 desenvolvedores**: Ana, Bruno, Carlos, Diego, Eduardo, Fábio e Gabriel. Destes, **Ana, Bruno e Carlos são seniores**; os demais são juniores.

Os projetos ativos são:

| Projeto | Tipo | Observação |
|---------|------|------------|
| **Orion** | BAU (manutenção) | — |
| **Lyra** | BAU (manutenção) | — |
| **Hydra** | Desenvolvimento | Requer ao menos 2 devs |
| **Vela** | Desenvolvimento | — |
| **Cygnus** | Desenvolvimento | Iniciado por Ana |

Projetos do tipo **BAU** são de manutenção contínua; projetos de **desenvolvimento** exigem foco total — um desenvolvedor não pode estar em dois projetos de desenvolvimento ao mesmo tempo.

---
## Variáveis de Decisão

Defina uma variável binária para cada par (desenvolvedor, projeto):

$$x_{d,p} \in \{0, 1\} \quad \forall\, d \in D,\; p \in P$$

onde $x_{d,p} = 1$ indica que o desenvolvedor $d$ está alocado ao projeto $p$.

---
## Função Objetivo

Maximizar a produtividade total da alocação:

$$\text{Maximizar} \quad \sum_{d \in D} \sum_{p \in P} \text{prod}_{d,p} \cdot x_{d,p}$$

onde $\text{prod}_{d,p}$ é a pontuação de compatibilidade do desenvolvedor $d$ com o projeto $p$ (valores de 1 a 10).

---
## Restrições

1. **Binariedade:** $x_{d,p} \in \{0,1\}$ para todo $d \in D$, $p \in P$.

2. **Exclusividade em desenvolvimento:** nenhum dev pode estar em mais de um projeto de desenvolvimento simultaneamente:
$$x_{d,\text{Hydra}} + x_{d,\text{Vela}} + x_{d,\text{Cygnus}} \leq 1 \quad \forall\, d \in D$$

3. **Sênior obrigatório em desenvolvimento:** cada projeto de desenvolvimento deve ter pelo menos um dev sênior:
$$\sum_{s \in \{\text{Ana, Bruno, Carlos}\}} x_{s,p} \geq 1 \quad \forall\, p \in \{\text{Hydra, Vela, Cygnus}\}$$

4. **Tamanho mínimo do Hydra:** o projeto Hydra deve ter pelo menos 2 desenvolvedores:
$$\sum_{d \in D} x_{d,\text{Hydra}} \geq 2$$

5. **Iniciador do Cygnus:** Ana deve estar alocada ao Cygnus (ela iniciou o projeto):
$$x_{\text{Ana},\text{Cygnus}} = 1$$

6. **Limites por projeto:** cada projeto deve ter entre $\min_p$ e $\max_p$ desenvolvedores alocados:
$$\min_p \leq \sum_{d \in D} x_{d,p} \leq \max_p \quad \forall\, p \in P$$

---
## Dados do Problema

### Desenvolvedores e senioridade

| Desenvolvedor | Nível |
|---------------|-------|
| Ana | Sênior |
| Bruno | Sênior |
| Carlos | Sênior |
| Diego | Júnior |
| Eduardo | Júnior |
| Fábio | Júnior |
| Gabriel | Júnior |

### Limites de alocação por projeto

| Projeto | Mín. devs | Máx. devs |
|---------|-----------|----------|
| Orion | 1 | 3 |
| Lyra | 1 | 2 |
| Hydra | 2 | 4 |
| Vela | 1 | 3 |
| Cygnus | 2 | 3 |

### Matriz de produtividade $\text{prod}_{d,p}$ (pontuação de 1 a 10)

| Dev \ Projeto | Orion | Lyra | Hydra | Vela | Cygnus |
|---------------|-------|------|-------|------|--------|
| Ana     | 5 | 10 | 7 | 8  | 10 |
| Bruno   | 4 | 6  | 10 | 7 | 5  |
| Carlos  | 8 | 6  | 8 | 10 | 6  |
| Diego   | 5 | 6  | 7 | 5  | 3  |
| Eduardo | 6 | 7  | 5 | 8  | 3  |
| Fábio   | 7 | 8  | 8 | 6  | 4  |
| Gabriel | 4 | 6  | 3 | 6  | 8  |

---

In [9]:
prob = LpProblem("Alocacao_Devs", LpMaximize)

devs = ["Ana", "Bruno", "Carlos", "Diego", "Eduardo", "Fábio", "Gabriel"]
projetos = ["Orion", "Lyra", "Hydra", "Vela", "Cygnus"]
dev_proj = ["Hydra", "Vela", "Cygnus"]
seniors = ["Ana", "Bruno", "Carlos"]

prod = {
    "Ana":     {"Orion": 5, "Lyra": 10, "Hydra": 7,  "Vela": 8,  "Cygnus": 10},
    "Bruno":   {"Orion": 4, "Lyra": 6,  "Hydra": 10, "Vela": 7,  "Cygnus": 5},
    "Carlos":  {"Orion": 8, "Lyra": 6,  "Hydra": 8,  "Vela": 10, "Cygnus": 6},
    "Diego":   {"Orion": 5, "Lyra": 6,  "Hydra": 7,  "Vela": 5,  "Cygnus": 3},
    "Eduardo": {"Orion": 6, "Lyra": 7,  "Hydra": 5,  "Vela": 8,  "Cygnus": 3},
    "Fábio":   {"Orion": 7, "Lyra": 8,  "Hydra": 8,  "Vela": 6,  "Cygnus": 4},
    "Gabriel": {"Orion": 4, "Lyra": 6,  "Hydra": 3,  "Vela": 6,  "Cygnus": 8},
}

limites = {
    "Orion":  (1, 3),
    "Lyra":   (1, 2),
    "Hydra":  (2, 4),
    "Vela":   (1, 3),
    "Cygnus": (2, 3),
}

x = LpVariable.dicts("x", (devs, projetos), cat="Binary")

# Objetivo
prob += lpSum(prod[d][p] * x[d][p] for d in devs for p in projetos)

# Restrição 2 — exclusividade em desenvolvimento
for d in devs:
    prob += lpSum(x[d][p] for p in dev_proj) <= 1

# Restrição 3 — sênior obrigatório em cada projeto de desenvolvimento
for p in dev_proj:
    prob += lpSum(x[s][p] for s in seniors) >= 1

# Restrição 4 — Hydra precisa de pelo menos 2 devs
prob += lpSum(x[d]["Hydra"] for d in devs) >= 2

# Restrição 5 — Ana deve estar no Cygnus
prob += x["Ana"]["Cygnus"] == 1

# Restrição 6 — limites min/max por projeto
for p in projetos:
    mn, mx = limites[p]
    prob += lpSum(x[d][p] for d in devs) >= mn
    prob += lpSum(x[d][p] for d in devs) <= mx

prob.solve(PULP_CBC_CMD(msg=0))

print(f"Status: {LpStatus[prob.status]}")
print(f"Produtividade total: {value(prob.objective):.0f}\n")

for d in devs:
    alocados = [p for p in projetos if value(x[d][p]) == 1]
    print(f"  {d:<10} → {', '.join(alocados) if alocados else '— sem alocação'}")

print("\n" + "="*40)
for p in projetos:
    alocados = [d for d in devs if value(x[d][p]) == 1]
    print(f"\n  📁 {p}")
    for d in alocados:
        nivel = "🔵 Sênior" if d in seniors else "⚪ Júnior"
        print(f"     {nivel}  {d}")

print("\n" + "="*40)
for p in projetos:
    alocados = [d for d in devs if value(x[d][p]) == 1]
    prod_proj = sum(prod[d][p] for d in alocados)
    tipo = "🔧 BAU" if p in ["Orion", "Lyra"] else "🚀 Dev"
    print(f"\n  {tipo}  {p}  ({len(alocados)} devs | prod: {prod_proj})")
    for d in alocados:
        nivel = "🔵 Sênior" if d in seniors else "⚪ Júnior"
        print(f"     {nivel}  {d:<10}  prod: {prod[d][p]}")

Status: Optimal
Produtividade total: 100

  Ana        → Lyra, Cygnus
  Bruno      → Hydra
  Carlos     → Orion, Vela
  Diego      → Hydra
  Eduardo    → Orion, Vela
  Fábio      → Orion, Lyra, Hydra
  Gabriel    → Cygnus


  📁 Orion
     🔵 Sênior  Carlos
     ⚪ Júnior  Eduardo
     ⚪ Júnior  Fábio

  📁 Lyra
     🔵 Sênior  Ana
     ⚪ Júnior  Fábio

  📁 Hydra
     🔵 Sênior  Bruno
     ⚪ Júnior  Diego
     ⚪ Júnior  Fábio

  📁 Vela
     🔵 Sênior  Carlos
     ⚪ Júnior  Eduardo

  📁 Cygnus
     🔵 Sênior  Ana
     ⚪ Júnior  Gabriel


  🔧 BAU  Orion  (3 devs | prod: 21)
     🔵 Sênior  Carlos      prod: 8
     ⚪ Júnior  Eduardo     prod: 6
     ⚪ Júnior  Fábio       prod: 7

  🔧 BAU  Lyra  (2 devs | prod: 18)
     🔵 Sênior  Ana         prod: 10
     ⚪ Júnior  Fábio       prod: 8

  🚀 Dev  Hydra  (3 devs | prod: 25)
     🔵 Sênior  Bruno       prod: 10
     ⚪ Júnior  Diego       prod: 7
     ⚪ Júnior  Fábio       prod: 8

  🚀 Dev  Vela  (2 devs | prod: 18)
     🔵 Sênior  Carlos      prod: 10
  